In [1]:
import torch
import torch.nn as nn
import numpy as np

In [4]:
class SimpleEncoder(nn.Module):
	def __init__(self, vocab: dict[str, float], embedding_dim: int, max_seq_len: int):
		super(SimpleEncoder, self).__init__()
		self.vocab = vocab
		self.embedding_dim = embedding_dim
		self.max_seq_len = max_seq_len

		self.word_embedding = nn.Embedding(len(vocab), embedding_dim)
		self.position_embedding = nn.Embedding(max_seq_len, embedding_dim)

	def load_glove(self, glove_path: str):
		with open(glove_path, 'r', encoding='utf-8') as f:
			for line in f:
				word, *vector = line.split()
				if word in self.vocab:
					self.word_embedding.weight.data[self.vocab[word]] = torch.tensor(list(map(float, vector)))

	def forward(self, x: torch.Tensor):
		# x: (batch_size, seq_len)
		batch_size, seq_len = x.size()
		positions = torch.arange(seq_len).repeat(batch_size, 1).to(x.device)
		# positions: (batch_size, seq_len)
		embeddings = self.word_embedding(x) + self.position_embedding(positions)
		# embeddings: (batch_size, seq_len, embedding_dim)
		return embeddings

In [6]:
from scipy.special import softmax
import numpy as np

In [34]:
import torch

# Generate a random number between 0 and 9
random_number = torch.randint(low=0, high=10, size=(1,))
print(random_number.item())

In [93]:
'''
Input embeddings of tokens with positional encoding
Input Matrix: seq_len x d_model
'''
N, d_model, max_seq_len = 10, 512, 100
inputs = [torch.randn(np.random.randint(1, max_seq_len), d_model, dtype=torch.float32) for _ in range(N)]


In [91]:
'''
Linear Transformation of input embeddings to obtain Query, Key and Value matrices
'''
heads = 8
d_k = d_model // heads

w_query = torch.randn(d_model, d_k, dtype=torch.float32)
w_key = torch.randn(d_model, d_k, dtype=torch.float32)
w_value = torch.randn(d_model, d_k, dtype=torch.float32)

# Scaled Dot-Product Attention
def self_attention(query, key, value):
	output = torch.matmul(query, key.T) / np.sqrt(d_k)
	output = torch.softmax(output, axis=1)
	output = torch.matmul(output, value)
	return output

for input in inputs:
	output = []
	for _ in range(heads):
		query = torch.matmul(input, w_query)
		key = torch.matmul(input, w_key)
		value = torch.matmul(input, w_value)
		output.append(self_attention(query, key, value))

	output = torch.cat(output, dim=1)
	print(output.shape)

torch.Size([45, 512])
torch.Size([74, 512])
torch.Size([73, 512])
torch.Size([11, 512])
torch.Size([94, 512])
torch.Size([83, 512])
torch.Size([41, 512])
torch.Size([33, 512])
torch.Size([64, 512])
torch.Size([51, 512])


In [ ]:
transient and complex